# BHR Anomaly Detection - Autoencoder Training

This notebook trains an autoencoder on **normal** router traffic data (probability=0) to detect Black Hole Router (BHR) attacks.

**Steps:**
1. Upload `anomaly_features_p0.csv` (normal traffic)
2. Train the autoencoder
3. Download `bhr_autoencoder.pth` for local inference

In [ ]:
# Cell 1: Check GPU and imports
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from google.colab import files

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

In [ ]:
# Cell 2: Define the Autoencoder Model
FEATURE_COLUMNS = [
    'flit_in', 'flit_out', 'avg_wait', 'max_wait', 'buffer_occ', 
    'active_vcs', 'stalls', 'credits', 'crossbar', 'io_ratio',
    'sw_in_arb', 'sw_out_arb', 'empty_vcs', 'total_wait', 
    'min_cred', 'max_cred', 'credit_sends'
]
NUM_FEATURES = len(FEATURE_COLUMNS)
print(f"Number of features: {NUM_FEATURES}")

class BHRAutoencoder(nn.Module):
    def __init__(self, input_dim=NUM_FEATURES, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 12), nn.ReLU(), nn.BatchNorm1d(12),
            nn.Linear(12, 8), nn.ReLU(), nn.BatchNorm1d(8),
            nn.Linear(8, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8), nn.ReLU(), nn.BatchNorm1d(8),
            nn.Linear(8, 12), nn.ReLU(), nn.BatchNorm1d(12),
            nn.Linear(12, input_dim)
        )
    
    def forward(self, x):
        return self.decoder(self.encoder(x))
    
    def get_error(self, x):
        with torch.no_grad():
            return torch.mean((x - self.forward(x)) ** 2, dim=1)

print("Model defined: 17 -> 12 -> 8 -> 4 -> 8 -> 12 -> 17")

In [ ]:
# Cell 3: Upload Training Data
print("Upload anomaly_features_p0.csv (NORMAL traffic, probability=0)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"\nUploaded: {filename}")

In [ ]:
# Cell 4: Load and Preprocess Data
df = pd.read_csv(filename)
df = df.replace([np.inf, -np.inf], np.nan).dropna()
print(f"Loaded {len(df)} samples from {df['router_id'].nunique()} routers")

X = df[FEATURE_COLUMNS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Val split
n_val = int(len(X_scaled) * 0.2)
idx = np.random.permutation(len(X_scaled))
X_train = torch.FloatTensor(X_scaled[idx[n_val:]]).to(device)
X_val = torch.FloatTensor(X_scaled[idx[:n_val]]).to(device)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

In [ ]:
# Cell 5: Train the Model
model = BHRAutoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

epochs, batch_size = 100, 256
history = {'train': [], 'val': []}
best_loss, best_state = float('inf'), None

for epoch in range(epochs):
    model.train()
    losses = []
    for i in range(0, len(X_train), batch_size):
        batch = X_train[i:i+batch_size]
        optimizer.zero_grad()
        loss = criterion(model(batch), batch)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val), X_val).item()
    
    history['train'].append(np.mean(losses))
    history['val'].append(val_loss)
    
    if val_loss < best_loss:
        best_loss, best_state = val_loss, model.state_dict().copy()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: Train={np.mean(losses):.6f}, Val={val_loss:.6f}")

model.load_state_dict(best_state)
print(f"\n✓ Best validation loss: {best_loss:.6f}")

In [ ]:
# Cell 6: Calculate Threshold and Plot
model.eval()
errors = model.get_error(X_train).cpu().numpy()
threshold = np.mean(errors) + 3 * np.std(errors)
print(f"Anomaly Threshold: {threshold:.6f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train'], label='Train')
ax1.plot(history['val'], label='Val')
ax1.set_title('Loss'); ax1.legend()

ax2.hist(errors, bins=50, alpha=0.7)
ax2.axvline(threshold, color='r', linestyle='--', label=f'Threshold={threshold:.4f}')
ax2.set_title('Reconstruction Errors'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7: Save and Download Model
save_dict = {
    'model_state': model.state_dict(),
    'scaler': scaler,
    'threshold': threshold
}
torch.save(save_dict, 'bhr_autoencoder.pth')
print("Model saved!")

files.download('bhr_autoencoder.pth')
print("\n✅ Download complete! Copy bhr_autoencoder.pth to gem5/anomaly_detection/")

In [ ]:
# Cell 8 (Optional): Test on Attack Data
print("Optional: Upload attack data to test (e.g., anomaly_features_p0.1.csv)")
try:
    up2 = files.upload()
    test_df = pd.read_csv(list(up2.keys())[0]).replace([np.inf, -np.inf], np.nan).dropna()
    X_test = scaler.transform(test_df[FEATURE_COLUMNS].values)
    test_errors = model.get_error(torch.FloatTensor(X_test).to(device)).cpu().numpy()
    test_df['anomaly'] = test_errors > threshold
    
    print("\n=== Results by Router ===")
    print(test_df.groupby('router_id')['anomaly'].agg(['sum', 'mean']).round(3))
    
    r10 = test_df[test_df['router_id'] == 10]
    print(f"\n⚠️ Router 10 (BHR): {r10['anomaly'].mean():.1%} anomaly rate")
except:
    print("Skipped test.")